# 4.8 增加深度改變了什麼？


一輪討論後，人會帶著已有認識再讀一次資料；模型堆更多block，也讓當前位置表示反覆讀前文、處理特徵、加上修正。這個重複次數叫深度（depth）。第二層讀的是第一層更新後的表示，不是把同樣的原卡重複平均一次，所以它有機會建立更複雜的關係。但機會、參數成本與實際學會多少，必須分開看。

[4.5](https://birdhackor.github.io/tiny-perceptron-vlm/4.5.html) 的block輸入輸出形狀一致，因此同寬度的block可以依序接好。整個模型還需要輸入字表、位置表、最後正規化與輸出層；增加block時，這些入口出口不會每層複製一次。字表是[2.1的查表特徵](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html)：本例20個字編號各有一列8個可調數字。位置表是[4.1的位置提示](https://birdhackor.github.io/tiny-perceptron-vlm/4.1.html)：最多128個位置各有一列8個可調數字，查出後與字特徵相加。兩張表每列都寬8，才能接到每位置8特徵的block。於是兩層模型的參數總數通常不等於一層的兩倍。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.model import TinyLM, ModelConfig

for depth in [1, 2, 3]:
    model = TinyLM(ModelConfig(vocab_size=20, width=8, layers=depth))
    count = model.description()["parameters"]
    logits = model(torch.tensor([[1, 2]]))["logits"]
    print("層數", depth, "參數格數", count, "輸出", logits.shape)
    assert logits.shape == (1, 2, 20)

輸入模型設定字表20、每位置8特徵，深度分別1、2、3；`[[1,2]]`是一筆含兩個字編號的輸入，模型先查表，再讓兩位置依序通過各block。最後[4.6的輸出層](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)把每位置8個內部特徵轉成20個候選字的原始分數logits，尚未轉成機率或生成文字。因此輸出三軸是筆數1、位置數2、候選數20；最後一軸由字表大小決定，不是模型寬度。`.description()` 返回設置與總參數格數，計的是所有可調表格裡有多少數，不是文件多少行或執行耗時。這個預設位置表上限128的例子，三個參數數量應為2200、3040、3880；每新增一block增加840，輸出始終 `[1,2,20]`。

可以把總數拆成「固定入口出口1360 + 深度×每block840」。固定項包括20×8的字向量、128×8的位置向量、最後LN的縮放與偏移、以及8×20的輸出權重。block裡的Q/K/V與輸出配方、FFN、兩次LN則每層各有自己的參數。這個拆法解釋了為什麼三層並非2200乘3，也讓你知道限制模型預算時哪些部分會跟着深度改變。

更深通常也增加每次前向與求導的運算，並保存更多中間材料。寬度增加則影響許多矩陣的兩邊，像8×8變成16×16是四倍，不只是兩倍；字表與位置表的成本才大致隨寬度線性增加。因而不能把深度、寬度、字表三種變化都用相同的倍率估算。

這段模型全部未訓練，隨機輸出或初始loss只說明設定建立成功，不足以證實深層模型更準。公平比較要用相同資料，說清訓練步數或總計算預算，並按 [1.2](https://birdhackor.github.io/tiny-perceptron-vlm/1.2.html) 在留出的資料評估。更大模型也可能因資料太少只記住訓練句，深度不是保證。

練習只把width從8改成16，其他固定，先預測三個輸出仍 `[1,2,20]`，參數量會增但不是一律兩倍，再執行核對。記錄每加一層的差值，對比原先840；若想判斷品質，則另安排訓練實驗，不要把這份結構檢查當能力排行榜。
